In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

In [0]:
dados_brutos = (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("quote", '"')
        .option("escape", '"')
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .load(CAMINHO)
    )

dados_brutos.columns

## Renomeação de colunas

Padronização dos nomes das colunas originais do arquivo bruto para o formato **snake_case**, removendo caracteres especiais, acentos e espaços. Esse processo garante consistência técnica para o ecossistema PySpark/Delta Lake.

In [0]:
colunas_renomeadas = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
}

dados_brutos_atualizados = dados_brutos.select(
    [F.col(coluna).alias(colunas_renomeadas[coluna]) for coluna in dados_brutos.columns]
)
display(dados_brutos)

## Auditoria

Inclusão de colunas técnicas de linhagem para garantir a rastreabilidade e governança do pipeline na camada Bronze:
* `_arquivo_origem`: Identifica o arquivo CSV exato de onde o registro foi extraído.
* `_ingerido_em`: Registra o timestamp preciso do momento do processamento.

In [0]:
bronze = dados_brutos_atualizados.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

## Persistência no Delta Lake (Carga Full Refresh)
Gravação do DataFrame no formato **Delta Lake**, aplicando a estratégia de recarga completa (*overwrite*):
* **`mode("overwrite")`**: Substitui o conteúdo existente pelos novos dados processados.
* **`option("overwriteSchema", "true")`**: Permite a atualização automática do esquema da tabela caso novas colunas sejam adicionadas futuramente.

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

## Gestão de metadados

Documentação e verificação da qualidade da carga executada na tabela Bronze:
* **Documentação do Catálogo**: Adição do comentário descritivo no Unity Catalog/Hive Metastore informando a regra de negócio, origem e histórico dos dados.
* **Validação de Ingestão**: Consulta de agregação para conferir a contagem total de registros e o horário da última carga por arquivo de origem.

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 - jul/2026).
          Dados brutos: todas as colunas como String, nenhuma linha removida.
          Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra'
          """)

In [0]:
display(
    spark.sql(f"""
              SELECT _arquivo_origem, count(*) AS linhas, max(_ingerido_em) AS ingerido_em
              FROM {TABELA}
              GROUP BY _arquivo_origem
              ORDER BY _arquivo_origem
              """)
)